<a href="https://colab.research.google.com/github/cmeli4/AI-Voice-Solutions/blob/main/Lesson_6_Automatic_Speech_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lección 6: Reconocimiento automático del habla

En el aula, las bibliotecas ya están instaladas para usted.

Si desea ejecutar este código en su propia máquina, puede instalar lo siguiente:

In [ ]:
!pip install transformers
!pip install -U datasets
!pip install soundfile
!pip install librosa
!pip install gradio

Es posible que la biblioteca librosa necesite tener instalado ffmpeg.


Esta página en librosa proporciona instrucciones de instalación para ffmpeg.


In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("librispeech_asr",
                       split="train.clean.100",
                       streaming=True,
                       trust_remote_code=True)

In [ ]:
example = next(iter(dataset))

In [ ]:
dataset_head = dataset.take(5)
list(dataset_head)

In [ ]:
list(dataset_head)[2]

In [ ]:
example

In [ ]:
from IPython.display import Audio as IPythonAudio

IPythonAudio(example["audio"]["array"],
             rate=example["audio"]["sampling_rate"])

Construir el canal

In [ ]:
from transformers import pipeline

In [ ]:
zero_shot_classifier = pipeline(
    task="zero-shot-audio-classification",
    model="laion/clap-htsat-unfused")

In [ ]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en")

In [ ]:
asr.feature_extractor.sampling_rate

In [ ]:
example['audio']['sampling_rate']

In [ ]:
asr = pipeline(task="automatic-speech-recognition",
               model="openai/whisper-tiny")

In [ ]:
asr(example["audio"]["array"])

In [ ]:
example["text"]

Crear una aplicación que se pueda compartir con Gradio

Sugerencia para solucionar problemas

Ten en cuenta que, en el aula, es posible que veas que el código para crear la aplicación Gradio se ejecuta indefinidamente.

Esto es específico de este entorno de aula cuando atiende a muchos alumnos a la vez, y no experimentarás este problema si ejecutas este código en tu propio equipo.

Para solucionarlo, reinicia el kernel (Menú Kernel->Restart Kernel) y vuelve a ejecutar el código en el laboratorio desde el principio de la lección.

Traducción realizada con la versión gratuita del traductor DeepL.com

In [ ]:
import os
import gradio as gr
demo = gr.Blocks()

In [ ]:
def transcribe_speech(filepath):
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        return ""
    output = asr(filepath)
    return output["text"]

In [ ]:
mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never")

Para aprender más sobre la creación de aplicaciones con Gradio, puedes consultar el curso corto: Creación de aplicaciones de IA generativa con Gradio, también impartido por Hugging Face.

In [ ]:
file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never",
)

In [ ]:
with gr.Blocks() as demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True)

with gr.Blocks() as demo:

    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(debug=True)

In [ ]:
demo.close()

Nota: Detenga la demostración antes de continuar con el resto del laboratorio.

* La aplicación seguirá ejecutándose a menos que ejecutes

demo.close()



* Si ejecutas otra aplicación gradio (más adelante en esta lección) sin cerrar primero esta aplicación, verás un mensaje de error:

OSError: Cannot find empty port in range

* Probar con un archivo de audio más largo

In [ ]:
! add-apt-repository -y ppa:savoury1/ffmpeg4
! apt-get -qq install -y ffmpeg

In [ ]:
# on Ubuntu or Debian
!sudo apt update && sudo apt install ffmpeg

In [ ]:
import soundfile as sf
import io
audio, sampling_rate = sf.read('/content/20241205-123416-654494046-921285621.mp3') #/content/20241205-123416-654494046-921285621.mp3

In [ ]:
sampling_rate

In [ ]:
asr.feature_extractor.sampling_rate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Nueva sección

# Nueva sección

In [ ]:
asr(audio)

In [ ]:
asr(audio, return_timestamps=True)

In [ ]:
audio.shape

In [ ]:
import numpy as np

audio_transposed = np.transpose(audio)

In [ ]:
audio_transposed.shape

In [ ]:
import librosa

In [ ]:
audio_mono = librosa.to_mono(audio_transposed)

In [ ]:
IPythonAudio(audio_mono,
             rate=sampling_rate)

In [ ]:
asr(audio_mono)

In [ ]:
sampling_rate

In [ ]:
asr.feature_extractor.sampling_rate

In [ ]:
audio_16KHz = librosa.resample(audio_mono,
                               orig_sr=sampling_rate,
                               target_sr=16000)

In [ ]:
asr(
    audio_16KHz,
    chunk_length_s=30, # 30 seconds
    batch_size=4,
    return_timestamps=True,
)["chunks"]

In [ ]:
import gradio as gr
demo = gr.Blocks()

In [ ]:
def transcribe_long_form(filepath):
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        return ""
    output = asr(
      filepath,
      max_new_tokens=256,
      chunk_length_s=30,
      batch_size=8,
    )
    return output["text"]

In [ ]:
mic_transcribe = gr.Interface(
    fn=transcribe_long_form,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never")

file_transcribe = gr.Interface(
    fn=transcribe_long_form,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
    allow_flagging="never",
)

In [ ]:
with gr.Blocks() as demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )
demo.launch(share=True)